# Module 15: End-to-End Scenario — Knowledge Retrieval (Feast + RAG)

## What You'll Learn

- Feast as unified retrieval layer for **existing Feast adopters** who also need RAG
- Based on Scenario B from the data strategy proposal (Feast variant path)
- Steps: Process documents → Generate embeddings → Store in vector-enabled Feast → Retrieve for RAG
- Combined retrieval: structured ML features + document embeddings
- Why the primary RAG stack on RHOAI is **OGX + Milvus** (not Feast)
- When to use Feast for RAG vs when to use the non-Feast stack

## Prerequisites

- Module 08 (RAG & Vector Search) concepts
- Upstream Feast with vector support (not RHOAI downstream)
- `feast`, `sentence-transformers`, `pandas` installed

---

> **📍 RHOAI STATUS**: Feast RAG is **NOT downstream**. This entire module uses **upstream-only** features.
>
> **🔮 UPSTREAM**: All of this is **Alpha upstream** — API may change. `retrieve_online_documents_v2`, `DocEmbedder`, vector-enabled stores are not productized in RHOAI.
>
> **🗺️ DATA STRATEGY**: Pillar 3 resolved two workload patterns: **predictive AI** (Feast) and **knowledge retrieval** (OGX+Milvus). This shows the Feast variant for existing adopters who want a unified layer.
>
> **For most RHOAI users starting with GenAI/RAG, OGX + Milvus is the recommended path.** Feast RAG is the path for teams already invested in Feast who want to add RAG capability without standing up a separate vector stack.

## When to Use Which Stack

```
Decision Tree:

  Starting fresh with RAG/GenAI?
    └── YES → OGX + Milvus (primary RHOAI path)
              - Native vector search
              - Agent orchestration built-in
              - GA downstream in RHOAI

  Already running Feast for predictive AI?
    └── YES → Feast RAG variant (this module)
              - Unified retrieval layer
              - Add documents to existing feature store
              - Alpha upstream only

  Need BOTH predictive features AND document retrieval?
    └── Feast path: two API calls, application merges
    └── OGX path:   separate stacks, no unified layer
    └── Future:     Data Hub UI unifies discovery (not retrieval)
```

> **⚠️ GAP**: No combined ML features + RAG document retrieval in a **single API call**. Two retrieval paths needed: `get_online_features()` + `retrieve_online_documents_v2()`.

## Scenario Overview

A financial services team already uses Feast for credit scoring (Module 14). They want to add a **compliance knowledge base** so loan officers can retrieve relevant regulations during application review.

```
┌─────────────────┐    ┌──────────────┐    ┌─────────────────┐
│ Compliance Docs │───▶│ Chunk + Embed│───▶│ Vector Store    │
│ (PDFs, policies)│    │ (DocEmbedder)│    │ (pgvector/Milvus)│
└─────────────────┘    └──────────────┘    └─────────────────┘
                                                    │
┌─────────────────┐    ┌──────────────┐          │
│ Credit Features │───▶│ Online Store │◀─────────┘
│ (Module 14)     │    │ (unified)    │
└─────────────────┘    └──────────────┘
                                │
                    ┌───────────┴───────────┐
                    ▼                       ▼
           get_online_features()   retrieve_online_documents_v2()
           (credit score, DTI)      (compliance doc chunks)
                    │                       │
                    └───────────┬───────────┘
                                ▼
                         LLM Prompt Assembly
```

## Step 1: Process Sample Documents into Chunks

In [ ]:
import os
import pandas as pd
import numpy as np
from datetime import datetime

os.makedirs("data", exist_ok=True)

# Simulated compliance documents (in production: use Docling for PDF parsing)
compliance_docs = [
    {
        "doc_id": "REG-001",
        "title": "Fair Lending Act Compliance",
        "content": (
            "The Fair Lending Act requires financial institutions to evaluate credit applications "
            "without discrimination based on race, color, religion, national origin, sex, "
            "marital status, or age. All scoring models must be regularly tested for disparate impact. "
            "Documentation of model decisions must be retained for 7 years."
        ),
    },
    {
        "doc_id": "REG-002",
        "title": "Debt-to-Income Ratio Guidelines",
        "content": (
            "Maximum debt-to-income ratio for unsecured personal loans is 43%. "
            "For secured loans, the limit may be extended to 50% with additional collateral. "
            "DTI must be calculated using verified income documentation, not self-reported values. "
            "Exceptions require senior underwriter approval and must be logged."
        ),
    },
    {
        "doc_id": "REG-003",
        "title": "Credit Score Minimum Requirements",
        "content": (
            "Minimum bureau score for standard approval is 620. Scores between 580-619 require "
            "manual review with compensating factors. Scores below 580 are auto-declined unless "
            "secured by collateral exceeding 150% of loan value. FICO and VantageScore are both accepted."
        ),
    },
    {
        "doc_id": "REG-004",
        "title": "Anti-Money Laundering Procedures",
        "content": (
            "All loan applications exceeding $10,000 must undergo enhanced due diligence. "
            "Source of funds must be verified for amounts over $25,000. Suspicious activity "
            "must be reported to the compliance officer within 24 hours. Annual AML training is mandatory."
        ),
    },
    {
        "doc_id": "REG-005",
        "title": "Payment History Evaluation Standards",
        "content": (
            "Payment history is weighted at 35% of the composite risk score. More than 2 missed "
            "payments in the last 12 months triggers mandatory review. On-time payment ratio "
            "above 95% qualifies for rate reduction. Bankruptcy within 7 years requires special handling."
        ),
    },
]

# Chunk documents (simple sentence-based chunking; production uses Docling)
CHUNK_SIZE = 200
chunks = []
for doc in compliance_docs:
    words = doc["content"].split()
    for i in range(0, len(words), CHUNK_SIZE // 10):
        chunk_text = " ".join(words[i:i + CHUNK_SIZE // 10])
        if len(chunk_text) > 20:
            chunks.append({
                "doc_id": doc["doc_id"],
                "title": doc["title"],
                "chunk_id": f"{doc['doc_id']}-chunk-{i // (CHUNK_SIZE // 10)}",
                "content": chunk_text,
                "event_timestamp": datetime(2024, 6, 1),
            })

chunks_df = pd.DataFrame(chunks)
chunks_df.to_parquet("data/compliance_chunks.parquet")
print(f"Created {len(chunks_df)} chunks from {len(compliance_docs)} documents")
chunks_df.head(3)

## Step 2: Generate Embeddings

In [ ]:
# Generate embeddings using sentence-transformers
# In production with large corpora: use DocEmbedder (v0.62.0) or Ray-distributed generation

try:
    from sentence_transformers import SentenceTransformer

    model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
    embeddings = model.encode(chunks_df["content"].tolist(), show_progress_bar=True)
    chunks_df["embedding"] = list(embeddings)
    print(f"Generated {len(embeddings)} embeddings (dim={embeddings.shape[1]})")
except ImportError:
    print("sentence-transformers not installed — generating mock embeddings")
    EMBEDDING_DIM = 384
    np.random.seed(42)
    chunks_df["embedding"] = [
        np.random.randn(EMBEDDING_DIM).astype(np.float32).tolist()
        for _ in range(len(chunks_df))
    ]
    print(f"Generated {len(chunks_df)} mock embeddings (dim={EMBEDDING_DIM})")

chunks_df.to_parquet("data/compliance_chunks_embedded.parquet")
print(f"Saved embedded chunks to data/compliance_chunks_embedded.parquet")

## Step 3: Store in Feast with Vector-Enabled Online Store

In [ ]:
import yaml
from datetime import timedelta
from feast import Entity, FeatureView, Field, FileSource
from feast.types import String, Int64
from feast.data_format import ParquetFormat

# Vector-enabled feature store config (upstream only)
rag_config = {
    "project": "compliance_rag",
    "provider": "local",
    "registry": "data/rag_registry.db",
    "offline_store": {"type": "duckdb"},
    "online_store": {
        "type": "postgres",
        "host": "localhost",
        "port": 5432,
        "database": "feast_online",
        "user": "feast",
        "password": "feast",
        "vector_enabled": True,
        "vector_index_type": "ivfflat",
    },
}

os.makedirs("rag_feature_repo", exist_ok=True)
with open("rag_feature_repo/feature_store.yaml", "w") as f:
    yaml.dump(rag_config, f)

print("Created vector-enabled feature_store.yaml")
print("⚠️ vector_enabled requires pgvector extension — upstream Alpha feature")

In [ ]:
# Define document entity and vector feature view
document = Entity(name="document", join_keys=["chunk_id"])

compliance_source = FileSource(
    name="compliance_chunks_source",
    path=os.path.abspath("data/compliance_chunks_embedded.parquet"),
    timestamp_field="event_timestamp",
)

compliance_docs_fv = FeatureView(
    name="compliance_documents",
    entities=[document],
    ttl=timedelta(days=90),
    schema=[
        Field(name="doc_id", dtype=String),
        Field(name="title", dtype=String),
        Field(name="content", dtype=String),
        # Vector field — stored in vector-enabled online store
        Field(name="embedding", dtype=String),  # Serialized vector; actual type depends on backend
    ],
    source=compliance_source,
)

print("Defined entity: document (join_key=chunk_id)")
print("Defined feature view: compliance_documents (with embedding field)")

In [ ]:
# Apply and materialize (requires vector-enabled online store running)
try:
    from feast import FeatureStore

    store = FeatureStore(repo_path="rag_feature_repo")
    store.apply([document, compliance_docs_fv])
    print("feast apply complete")

    end = datetime.now()
    start = end - timedelta(days=30)
    store.materialize(start, end)
    print(f"Materialized document embeddings to online store")
except Exception as e:
    print(f"Materialization requires vector-enabled PostgreSQL: {e}")
    print("For local demo, we simulate retrieval below")

## Step 4: Retrieve Documents via Vector Similarity

In [ ]:
# Vector similarity retrieval using retrieve_online_documents_v2 (upstream Alpha API)

def cosine_similarity(a, b):
    a, b = np.array(a), np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def retrieve_documents_simulated(query: str, top_k: int = 3):
    """Simulate retrieve_online_documents_v2 when vector store is not available."""
    try:
        from sentence_transformers import SentenceTransformer
        model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
        query_embedding = model.encode(query)
    except ImportError:
        query_embedding = np.random.randn(384)

    scores = []
    for _, row in chunks_df.iterrows():
        emb = row["embedding"] if isinstance(row["embedding"], list) else row["embedding"].tolist()
        sim = cosine_similarity(query_embedding, emb)
        scores.append((row["chunk_id"], row["title"], row["content"], sim))

    scores.sort(key=lambda x: x[3], reverse=True)
    return scores[:top_k]

# Production API (upstream Alpha):
production_api = '''
# retrieve_online_documents_v2 — upstream Alpha API
results = store.retrieve_online_documents_v2(
    features=["compliance_documents"],
    query_embedding=query_embedding,
    top_k=3,
)
'''

query = "What are the debt-to-income ratio limits for loan approval?"
results = retrieve_documents_simulated(query, top_k=3)

print(f"Query: {query}\n")
print("Retrieved documents:")
for chunk_id, title, content, score in results:
    print(f"  [{score:.3f}] {title} ({chunk_id})")
    print(f"    {content[:120]}...\n")

## Step 5: Combine with Structured Features (Two-Call Pattern)

In [ ]:
# The two-call pattern: structured features + document retrieval
# Application code merges results before LLM prompt assembly

def combined_retrieval(customer_id: int, query: str, top_k: int = 3):
    """
    Two-call pattern for unified retrieval.
    Path 1: get_online_features() for credit scoring features
    Path 2: retrieve_online_documents_v2() for compliance docs
    """
    # --- Path 1: Structured ML features (from Module 14 credit scoring) ---
    structured_features = {
        "customer_id": customer_id,
        "bureau_score": 710,
        "debt_to_income": 0.38,
        "delinquencies_12m": 0,
        "payment_reliability": 0.92,
        "composite_risk_score": 0.15,
    }
    # In production:
    # structured = credit_store.get_online_features(
    #     features=[bureau_fv, payment_fv, credit_risk_odfv],
    #     entity_rows=[{"customer_id": customer_id}],
    # ).to_dict()

    # --- Path 2: Document retrieval (this module) ---
    documents = retrieve_documents_simulated(query, top_k=top_k)
    # In production:
    # documents = rag_store.retrieve_online_documents_v2(
    #     features=["compliance_documents"],
    #     query_embedding=embed(query),
    #     top_k=top_k,
    # )

    return structured_features, documents


customer_id = 42
query = "What compliance rules apply to this customer's debt-to-income ratio?"

features, docs = combined_retrieval(customer_id, query)

print("=== Path 1: Structured Features ===")
for k, v in features.items():
    print(f"  {k}: {v}")

print("\n=== Path 2: Retrieved Documents ===")
for chunk_id, title, content, score in docs:
    print(f"  [{score:.3f}] {title}")

print("\n=== Merged LLM Prompt ===")
prompt = f"""You are a loan compliance assistant.

Customer Profile:
- Bureau Score: {features['bureau_score']}
- Debt-to-Income: {features['debt_to_income']:.0%}
- Risk Score: {features['composite_risk_score']:.2f}
- Payment Reliability: {features['payment_reliability']:.0%}

Relevant Compliance Documents:
{chr(10).join(f'- {title}: {content[:100]}...' for _, title, content, _ in docs)}

Question: {query}

Provide a compliance assessment for this loan application."""

print(prompt[:500] + "...")

## Step 6: Trade-offs vs OGX + Milvus Path

### Comparison: Feast RAG vs OGX + Milvus

| Dimension | Feast RAG (this module) | OGX + Milvus (primary RHOAI) |
|-----------|------------------------|------------------------------|
| **RHOAI status** | NOT downstream (upstream Alpha) | GA downstream |
| **API stability** | Alpha — may change | GA — stable |
| **Vector search** | Via vector-enabled online store | Native Milvus integration |
| **Agent orchestration** | None — build your own | OGX agents, tools, workflows |
| **Unified retrieval** | Two-call pattern (features + docs) | Separate from Feast features |
| **Document processing** | Docling + DocEmbedder (upstream) | OGX document ingestion pipeline |
| **Best for** | Existing Feast adopters adding RAG | New GenAI/RAG workloads |
| **Compute** | Ray-distributed embeddings (manual) | OGX-managed inference |
| **UI** | No vector search in Feast UI | Milvus Attu + OGX dashboard |

### When Feast RAG Makes Sense

1. **Already invested in Feast** — credit scoring, fraud detection, or other predictive AI workloads running
2. **Want unified serving layer** — one online store for both tabular features and document embeddings
3. **Can accept Alpha risk** — upstream-only, API may change, no RHOAI operator support
4. **Small-to-medium document corpora** — pgvector works well; Milvus via Feast for larger scale

### When OGX + Milvus Is Better

1. **Starting fresh with GenAI/RAG** — no existing Feast deployment
2. **Need agent orchestration** — multi-step reasoning, tool use, workflow management
3. **Production requirements** — GA downstream, operator-managed, supported
4. **Large-scale vector search** — Milvus is purpose-built for billion-scale embeddings

> **🗺️ DATA STRATEGY**: The strategy resolved two paths, not one. Predictive AI → Feast. Knowledge retrieval → OGX+Milvus. This module shows the overlap zone for teams that need both.

In [ ]:
# OGX + Milvus equivalent (primary RHOAI path) — for comparison
ogx_milvus_pattern = '''
# OGX + Milvus path (GA downstream in RHOAI):

# 1. Ingest documents via OGX document pipeline
from ogx import DocumentProcessor
processor = DocumentProcessor(chunk_size=512)
chunks = processor.process("compliance_docs/")

# 2. Generate embeddings via OGX inference
embeddings = ogx_client.embed(chunks, model="all-MiniLM-L6-v2")

# 3. Store in Milvus (operator-managed)
milvus_client.insert(collection="compliance", vectors=embeddings, metadata=chunks)

# 4. Retrieve at query time via OGX agent
agent = ogx_client.create_agent(
    tools=["milvus_search", "feast_features"],  # Can call Feast for structured features
    system_prompt="You are a loan compliance assistant."
)
response = agent.query(
    "What compliance rules apply to customer 42 with DTI 38%?"
)

# OGX agent orchestrates:
#   - Milvus search for compliance docs
#   - Feast API call for customer features (if configured as tool)
#   - LLM reasoning over combined context
'''
print(ogx_milvus_pattern)
print("\n📍 RHOAI STATUS: OGX + Milvus is GA downstream — recommended for new RAG workloads")

## Summary

| Aspect | Feast RAG Path | OGX + Milvus Path |
|--------|---------------|-------------------|
| RHOAI status | Upstream Alpha only | GA downstream |
| Retrieval | Two API calls, app merges | Agent orchestrates |
| Vector store | pgvector/Milvus via Feast | Native Milvus |
| Best for | Existing Feast adopters | New GenAI workloads |
| Unified layer | Yes (one online store) | No (separate stacks) |

**Key takeaway**: Feast RAG is a viable path for teams already running Feast who want to add document retrieval without a second vector stack. For everyone else, **OGX + Milvus is the recommended RHOAI path** for knowledge retrieval.

---

**Tutorial complete.** Return to [README](../../README.md) or explore [docs/ui-guide.md](../../docs/ui-guide.md) for visual interfaces.